# Background (sky) trials on the corners and centre of a detector (v2: 5 trials, mesh sky)

Fits five cutouts -- the four corners and the centre of one LSSTCam detector, each a real tile
(~372 px, 30 px padding) -- with the fast preset, once per trial, and shows for every trial:

* a table: sky subtracted, median residual/sigma **between stars** (model < 0.5 sigma) and **on
  stars** (model > 4 sigma), robust residual scatter, number of sources, psfMag - Gaia G;
* per cutout, one row per trial: data | model | residual | residual/sigma | residual/sigma binned
  4x4 (as the whole-detector plot), with the same colour scales in every row.

The trials compare the background options; add your own to `TRIALS`.  Run with the **LSST** kernel.

In [ ]:
%matplotlib inline
import os
import subprocess
import sys
import time

# where this repository is checked out (default: the folder above this notebook)
MEAS_CROWD = os.environ.get("MEAS_CROWD_DIR") or os.path.abspath("..")
# MEAS_CROWD = "/sdf/data/rubin/user/abudlong/code/build/meas_crowd_v2"
PKG = os.path.join(MEAS_CROWD, "python")
if not os.path.isdir(os.path.join(PKG, "lsst", "meas", "crowd")):
    raise FileNotFoundError(f"no meas_crowd package under {PKG}: set MEAS_CROWD above")
if PKG not in sys.path:
    sys.path.insert(0, PKG)
import lsst  # noqa: E402
import lsst.meas  # noqa: E402
for mod, sub in ((lsst, "lsst"), (lsst.meas, os.path.join("lsst", "meas"))):
    if os.path.join(PKG, sub) not in list(mod.__path__):   # lsst / lsst.meas were imported before
        mod.__path__.append(os.path.join(PKG, sub))

In [ ]:
import glob

CHECK = """
import importlib.util, sys
spec = importlib.util.spec_from_file_location("_crowdLib", sys.argv[1])
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
print(hasattr(m, "solveSparseLeastSquares"))
"""

def solverBuilt():
    """True if the compiled extension has the sparse solver (checked in a fresh process)."""
    for so in glob.glob(os.path.join(PKG, "lsst", "meas", "crowd", "_crowdLib*.so")):
        out = subprocess.run([sys.executable, "-c", CHECK, so], capture_output=True, text=True)
        if out.stdout.strip() == "True":
            return True
    return False

if solverBuilt():
    print("C++ extension with the sparse solver: OK")
else:
    print("building the C++ extension ...")
    r = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=MEAS_CROWD,
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode == 0 and solverBuilt():
        print("\nBUILT -- now restart the kernel (Kernel > Restart) and run from the top.")
    else:
        raise RuntimeError("build failed: see the output above")

In [ ]:
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from scipy.spatial import cKDTree  # noqa: E402

import lsst.meas.crowd as crowd  # noqa: E402

print("meas_crowd from:", os.path.dirname(crowd.__file__))
print("C++ kernels:", crowd.HAVE_CXX, "| sparse Eigen solver:", crowd.HAVE_SPARSE_SOLVER)
assert crowd.HAVE_SPARSE_SOLVER, "the sparse solver is not built: run the build cell, restart"

# The notebook server's CPU limit is a quota (e.g. 4 CPUs) on a node whose cores are all visible
# (e.g. 128): without a cap, NumPy's BLAS starts a thread per visible core and the kernel is
# throttled.  Cap the threads at the usable CPUs.
crowd.limitBlasThreads()

## Helpers: the numbers and the figure of each trial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm


def _arrays(run):
    """Data, model, variance, bad mask and the subtracted sky of a run's final pass."""
    h = run.outerHistory[-1]
    exp, model = h["exposure"], np.asarray(h["result"]["modelFull"], float)
    data = np.asarray(exp.image.array, float)
    var = np.asarray(exp.variance.array, float)
    bad = ((exp.mask.array & run.badBitmask) != 0) | ~np.isfinite(data) | ~(var > 0)
    raw = run.rawCrop
    raw = raw.image.array if hasattr(raw, "image") else raw[exp.getBBox()].image.array
    return data, model, var, bad, np.asarray(raw, float) - data


def trialStats(run):
    """Numbers for one trial: sky subtracted, residual/sigma between and on stars, Gaia offset."""
    data, model, var, bad, sky = _arrays(run)
    z = (data - model) / np.sqrt(np.where(bad, np.inf, var))
    s = model / np.sqrt(np.where(bad, np.inf, var))
    ok = ~bad
    out = dict(sky=float(np.median(sky[ok])),
               between=float(np.median(z[ok & (s < 0.5)])),
               onStars=float(np.median(z[ok & (s > 4)])) if (ok & (s > 4)).any() else np.nan,
               robustSigma=float(1.4826 * np.median(np.abs(z[ok] - np.median(z[ok])))),
               nSources=len(run.outerHistory[-1]["result"]["peaks"]))
    ph = getattr(run, "photometry", None)
    if ph is not None and "gaiaDmag" in ph.colnames:
        g = np.asarray(ph["gaiaMatch"], bool) & (np.asarray(ph["psfSnr"]) >= 50) \
            & np.isfinite(np.asarray(ph["gaiaDmag"], float))
        d = np.asarray(ph["gaiaDmag"], float)[g]
        if len(d):
            out["gaiaDmag"] = float(np.median(d))
            out["gaiaScatter"] = float(1.4826 * np.median(np.abs(d - np.median(d))))
    return out


def binnedSigma(z, b=4):
    """Residual/sigma binned b x b as sum / sqrt(n) over the finite pixels (as plotDetector)."""
    ny, nx = (z.shape[0] // b) * b, (z.shape[1] // b) * b
    zz = z[:ny, :nx].reshape(ny // b, b, nx // b, b)
    fin = np.isfinite(zz)
    n = fin.sum(axis=(1, 3))
    tot = np.where(fin, zz, 0.0).sum(axis=(1, 3))
    return np.where(n > 0, tot / np.sqrt(np.maximum(n, 1)), np.nan)


def showTrials(name, runs, labels):
    """data | model | residual | residual/sigma | binned residual/sigma, one row per trial."""
    arrs = [_arrays(r) for r in runs]
    d0, m0, v0, b0, _ = arrs[0]
    good = ~b0
    lo, hi = np.percentile(d0[good], [1, 99.7])
    rs = 1.4826 * np.median(np.abs((d0 - m0)[good] - np.median((d0 - m0)[good])))
    dnorm = AsinhNorm(linear_width=max(3 * rs, 1e-9), vmin=lo, vmax=hi)
    fig, axes = plt.subplots(len(runs), 5, figsize=(23, 4.4 * len(runs)), squeeze=False)
    for row, ((data, model, var, bad, sky), run, lab) in enumerate(zip(arrs, runs, labels)):
        st = trialStats(run)
        resid = np.where(bad, np.nan, data - model)
        z = resid / np.sqrt(np.where(bad, np.nan, var))
        panels = [(np.where(bad, np.nan, data), dict(cmap="viridis", norm=dnorm), "data (sky-subtracted)"),
                  (np.where(bad, np.nan, model), dict(cmap="viridis", norm=dnorm), "model"),
                  (resid, dict(cmap="RdBu_r", vmin=-4 * rs, vmax=4 * rs), "residual"),
                  (z, dict(cmap="RdBu_r", vmin=-4, vmax=4), "residual / sigma"),
                  (binnedSigma(z), dict(cmap="RdBu_r", vmin=-4, vmax=4),
                   "residual / sigma, binned 4x4 (sum/sqrt n)")]
        for ax, (img, kw, title) in zip(axes[row], panels):
            ax.imshow(img, origin="lower", interpolation="nearest", **kw)
            ax.set_xticks([]), ax.set_yticks([])
            ax.set_title(title, fontsize=10)
            ax.set_facecolor("magenta")
        axes[row, 0].set_ylabel(lab, fontsize=11)
        axes[row, 3].set_title(f"residual / sigma: between stars {st['between']:+.3f}, "
                               f"on stars {st['onStars']:+.3f}", fontsize=9)
        axes[row, 2].set_title(f"residual (sky {st['sky']:+.2f} counts, robust sigma "
                               f"{st['robustSigma']:.3f}/px)", fontsize=9)
        axes[row, 1].set_title(f"model ({st['nSources']} sources)", fontsize=10)
    fig.suptitle(f"{name}: rows = trials; magenta = bad / unfitted pixels", fontsize=12)
    fig.tight_layout()
    return fig

## Settings, visit image, cutouts

In [ ]:
import contextlib, io, time  # noqa: E401

VISIT, DETECTOR = 2025062100433, 144
REPO, COLLECTION = "dp2_prep", "LSSTCam/runs/DRP/DP2"
SIZE, PAD = 372, 30
OUT = "sky_trials"
os.makedirs(OUT, exist_ok=True)

TRIALS = {                                   # label -> options on top of FAST_STELLAR_FIELD_CONFIG
    "no sky correction": dict(skyMedian=False, skyTerms=0),
    "median sky plane": dict(skyMedian="plane", skyTerms=0),
    "median sky mesh (fast default)": dict(skyMedian="mesh", skyTerms=0),
    "mesh + sky before 1st detection": dict(skyMedian="mesh", skyTerms=0, skyInitial=True),
    "mesh + deeper detection": dict(skyMedian="mesh", skyTerms=0, skyInitial=True,
                                    detectLocalBgPx=64, minSnr=3.0),
}

from lsst.daf.butler import Butler  # noqa: E402
butler = Butler(REPO, collections=COLLECTION)
refs = butler.query_datasets("visit_image", collections="LSSTCam/runs/*", find_first=False,
                             where=f"instrument='LSSTCam' AND visit={VISIT} AND detector={DETECTOR}")
visitImage = butler.get(refs[0])
bb = visitImage.getBBox()
W, H = bb.getWidth(), bb.getHeight()
CUTOUTS = {                                  # name -> lower-left corner of the cutout
    "bottom-left": (0, 0), "bottom-right": (W - SIZE, 0),
    "top-left": (0, H - SIZE), "top-right": (W - SIZE, H - SIZE),
    "centre": ((W - SIZE) // 2, (H - SIZE) // 2),
}
specs = [dict(x0=bb.getMinX() + x, y0=bb.getMinY() + y, w=SIZE, h=SIZE, name=n)
         for n, (x, y) in CUTOUTS.items()]
sess = crowd.prepareSession(VISIT, DETECTOR, specs, pad=PAD, psfSource="visit", visitImage=visitImage)
crowd.attachGaia(sess, visitImage=visitImage, repo=REPO)
print({n: f"x {x}-{x + SIZE - 1}, y {y}-{y + SIZE - 1}" for n, (x, y) in CUTOUTS.items()})

## Run every trial on every cutout

The pipeline's log of each fit is kept in `logs[(cutout, trial)]` (printed only if a fit fails).

In [ ]:
runs, logs, rows = {}, {}, []
for name in CUTOUTS:
    runs[name] = {}
    for label, opts in TRIALS.items():
        cfg = crowd.makeConfig(crowd.FAST_STELLAR_FIELD_CONFIG, **opts)
        buf = io.StringIO()
        t0 = time.perf_counter()
        with contextlib.redirect_stdout(buf):
            r = crowd.runSessionGaia(sess, cfg, names=[name], plotFinal=False)
        dt = time.perf_counter() - t0
        logs[(name, label)] = buf.getvalue()
        if name not in r:
            print(f"{name} / {label}: FAILED -- end of its log:\n" + "\n".join(buf.getvalue().splitlines()[-25:]))
            continue
        runs[name][label] = r[name]
        rows.append((name, label, dt, trialStats(r[name])))

hdr = (f"{'cutout':13s} {'trial':24s} {'time':>5s} {'sky':>7s} {'between':>8s} {'on stars':>8s} "
       f"{'scatter':>7s} {'sources':>7s} {'psfMag-G':>8s} {'(scatter)':>9s}")
print(hdr + "\n" + "-" * len(hdr))
for name, label, dt, st in rows:
    print(f"{name:13s} {label:24s} {dt:4.1f}s {st['sky']:+7.2f} {st['between']:+8.3f} "
          f"{st['onStars']:+8.3f} {st['robustSigma']:7.3f} {st['nSources']:7d} "
          f"{st.get('gaiaDmag', float('nan')):+8.3f} {st.get('gaiaScatter', float('nan')):9.3f}")
print("\nsky: counts subtracted (median); between / on stars: median residual/sigma where the model "
      "is < 0.5 / > 4 sigma (0 is ideal); scatter: robust residual/sigma per pixel (1 is ideal)")

## Data, model and residuals of every trial

One figure per cutout, one row per trial.  Data and model share a colour scale (asinh); residual
panels use the same range in every row, so differences between trials show directly.  Figures are
also saved to `sky_trials/`.

In [ ]:
for name in CUTOUTS:
    if not runs.get(name):
        continue
    labels = list(runs[name])
    fig = showTrials(f"{name} (x0, y0 = {CUTOUTS[name]})", [runs[name][k] for k in labels], labels)
    fig.savefig(os.path.join(OUT, f"{name}.png"), dpi=80)
    plt.show()